# InceptionV3

##  Step 1: Import Necessary libraries

In [15]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models

## Step 2: CHANGE ONLY THIS SECTION

In [16]:
IMAGE_SIZE = (224, 224)          # <- Change input size
BATCH_SIZE = 32
EPOCHS = 20

TRAIN_DIR = r"C:\Users\Abdul\Deep Learning\08_CNN\Image Classification\train_data"
VAL_DIR = r"C:\Users\Abdul\Deep Learning\08_CNN\Image Classification\val_data"
TEST_DIR=r"C:\Users\Abdul\Deep Learning\08_CNN\Image Classification\test_data"
NUM_CLASSES = 5                  # <- Number of classes

BASE_MODEL_NAME ="InceptionV3"   # <- Options: VGG16, ResNet50, MobileNetV2, etc

FREEZE_LAYERS = True             # <- Freeze base model
FINE_TUNE_AT = None              # <- Set layer index to unfreeze later

## Step 3: Data Preparation || Data Pipeline

In [17]:
train_datagen = ImageDataGenerator(rescale=1./255,
                                   horizontal_flip=True,
                                   zoom_range=0.2)

val_datagen = ImageDataGenerator(rescale=1./255)
test_generator = train_datagen.flow_from_directory(TEST_DIR,
                                               target_size=IMAGE_SIZE,
                                               batch_size=BATCH_SIZE,
                                               class_mode='categorical',
                                               shuffle=False)
train_data = train_datagen.flow_from_directory(TRAIN_DIR,
                                               target_size=IMAGE_SIZE,
                                               batch_size=BATCH_SIZE,
                                               class_mode='categorical')

val_data = val_datagen.flow_from_directory(VAL_DIR,
                                           target_size=IMAGE_SIZE,
                                           batch_size=BATCH_SIZE,
                                           class_mode='categorical')

Found 15 images belonging to 5 classes.
Found 50 images belonging to 5 classes.
Found 9 images belonging to 5 classes.


## Step 4: Model Building: Load PreTrained Model

In [18]:
def get_base_model(name):
    if name == "VGG16":
        return tf.keras.applications.VGG16(weights='imagenet',
                                           include_top=False,
                                           input_shape=(*IMAGE_SIZE, 3))
    elif name == "ResNet50":
        return tf.keras.applications.ResNet50(weights='imagenet',
                                              include_top=False,
                                              input_shape=(*IMAGE_SIZE, 3))
    elif name == "MobileNetV2":
        return tf.keras.applications.MobileNetV2(weights='imagenet',
                                                 include_top=False,
                                                 input_shape=(*IMAGE_SIZE, 3))
    elif name == "InceptionV3":
        return tf.keras.applications.InceptionV3(weights='imagenet',
                                                       include_top=False,
                                                       input_shape=(*IMAGE_SIZE, 3))

base_model = get_base_model(BASE_MODEL_NAME)

87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 8s 0us/step


### ====================================================================

# The Most Important 2 Steps in Transfer Learning: 
## 1. FREEZE BASE MODEL

In [19]:
if FREEZE_LAYERS:
    for layer in base_model.layers:
        layer.trainable = False

## 2. ADD CUSTOM HEAD

In [20]:
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)

outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = models.Model(inputs=base_model.input, outputs=outputs)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)    │ (None, 224, 224, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_203 (Conv2D)           │ (None, 111, 111, 32)      │             864 │ input_layer_1[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_203       │ (None, 111, 111, 32)      │              96 │ conv2d_203[0][0]           │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_203 (Activation)   │ (None, 111, 111, 32)      │               0 │ batch_normalization_203[0… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_204 (Conv2D)           │ (None, 109, 109, 32)      │           9,216 │ activation_203[0][0]       │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_204       │ (None, 109, 109, 32)      │              96 │ conv2d_204[0][0]           │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_204 (Activation)   │ (None, 109, 109, 32)      │               0 │ batch_normalization_204[0… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_205 (Conv2D)           │ (None, 109, 109, 64)      │          18,432 │ activation_204[0][0]       │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_205       │ (None, 109, 109, 64)      │             192 │ conv2d_205[0][0]           │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_205 (Activation)   │ (None, 109, 109, 64)      │               0 │ batch_normalization_205[0… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ max_pooling2d_4               │ (None, 54, 54, 64)        │               0 │ activation_205[0][0]       │
│ (MaxPooling2D)                │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_206 (Conv2D)           │ (None, 54, 54, 80)        │           5,120 │ max_pooling2d_4[0][0]      │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_206       │ (None, 54, 54, 80)        │             240 │ conv2d_206[0][0]           │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_206 (Activation)   │ (None, 54, 54, 80)        │               0 │ batch_normalization_206[0… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_207 (Conv2D)           │ (None, 52, 52, 192)       │         138,24

 Total params: 22,328,613 (85.18 MB)

 Trainable params: 525,829 (2.01 MB)

 Non-trainable params: 21,802,784 (83.17 MB)

### ====================================================================

### Step 4: Model Building Continues..It's COMPILATION Time.

In [21]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

## Step 5: Model Training | Model Evaluation | Model Testing

In [22]:
history = model.fit(train_data,
                    validation_data=val_data,
                    epochs=EPOCHS)

Epoch 1/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 11s 3s/step - accuracy: 0.1200 - loss: 2.1443 - val_accuracy: 0.2222 - val_loss: 1.7871
Epoch 2/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 935ms/step - accuracy: 0.2200 - loss: 2.0286 - val_accuracy: 0.0000e+00 - val_loss: 1.8368
Epoch 3/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 531ms/step - accuracy: 0.3000 - loss: 1.8265 - val_accuracy: 0.0000e+00 - val_loss: 1.8705
Epoch 4/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 537ms/step - accuracy: 0.2400 - loss: 1.8161 - val_accuracy: 0.0000e+00 - val_loss: 1.8516
Epoch 5/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 526ms/step - accuracy: 0.1800 - loss: 1.8805 - val_accuracy: 0.0000e+00 - val_loss: 1.8270
Epoch 6/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 558ms/step - accuracy: 0.2000 - loss: 1.6311 - val_accuracy: 0.0000e+00 - val_loss: 1.8022
Epoch 7/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 950ms/step - accuracy: 0.4200 - loss: 1.5030 - val_accuracy: 0.2222 - val_loss: 1.7762
Epoch 8/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 558ms/step - accuracy: 0.3600 - loss: 1.4774 - val_accuracy: 

##  (OPTIONAL Step): FINE-TUNING with new weights(NOT SUGGESTED)

In [23]:
if FINE_TUNE_AT is not None:
    for layer in base_model.layers[FINE_TUNE_AT:]:
        layer.trainable = True

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    print("Starting Fine-Tuning...")

    history_fine = model.fit(
        train_data,
        validation_data=val_data,
        epochs=5
    )

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)    │ (None, 224, 224, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_203 (Conv2D)           │ (None, 111, 111, 32)      │             864 │ input_layer_1[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_203       │ (None, 111, 111, 32)      │              96 │ conv2d_203[0][0]           │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_203 (Activation)   │ (None, 111, 111, 32)      │               0 │ batch_normalization_203[0… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_204 (Conv2D)           │ (None, 109, 109, 32)      │           9,216 │ activation_203[0][0]       │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_204       │ (None, 109, 109, 32)      │              96 │ conv2d_204[0][0]           │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_204 (Activation)   │ (None, 109, 109, 32)      │               0 │ batch_normalization_204[0… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_205 (Conv2D)           │ (None, 109, 109, 64)      │          18,432 │ activation_204[0][0]       │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_205       │ (None, 109, 109, 64)      │             192 │ conv2d_205[0][0]           │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_205 (Activation)   │ (None, 109, 109, 64)      │               0 │ batch_normalization_205[0… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ max_pooling2d_4               │ (None, 54, 54, 64)        │               0 │ activation_205[0][0]       │
│ (MaxPooling2D)                │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_206 (Conv2D)           │ (None, 54, 54, 80)        │           5,120 │ max_pooling2d_4[0][0]      │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_206       │ (None, 54, 54, 80)        │             240 │ conv2d_206[0][0]           │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_206 (Activation)   │ (None, 54, 54, 80)        │               0 │ batch_normalization_206[0… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_207 (Conv2D)           │ (None, 52, 52, 192)       │         138,24

 Total params: 23,380,273 (89.19 MB)

 Trainable params: 525,829 (2.01 MB)

 Non-trainable params: 21,802,784 (83.17 MB)

 Optimizer params: 1,051,660 (4.01 MB)

## Export the Intelligence File.

### Question: How to use this file for Prediction?

In [24]:
from tensorflow.keras.preprocessing import image
import numpy as np

# Load the image
img = image.load_img( r"C:\Users\Abdul\Deep Learning\08_CNN\Image Classification\test_data\Vijay\images (4).jpg", target_size=(224, 224))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = img_array / 255.0

# Predict
prediction = model.predict(img_array)

# Get predicted class index
predicted_index = np.argmax(prediction)

# Convert class index to person name
class_names = {v: k for k, v in test_generator.class_indices.items()}

# Print the person's name
print("Predicted Person:", class_names[predicted_index])

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Predicted Person: DQ


In [25]:
from tensorflow.keras.preprocessing import image
import numpy as np

# Load the image
img = image.load_img( r"C:\Users\Abdul\Deep Learning\08_CNN\Image Classification\test_data\DQ\images (12).jpg", target_size=(224, 224))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = img_array / 255.0

# Predict
prediction = model.predict(img_array)

# Get predicted class index
predicted_index = np.argmax(prediction)

# Convert class index to person name
class_names = {v: k for k, v in test_generator.class_indices.items()}

# Print the person's name
print("Predicted Person:", class_names[predicted_index])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step
Predicted Person: Dhruv


# THE END!